# QEvict on DeepSeek-V2-Lite-Chat: LongBench at a 10% KV-cache budget (Colab)

This notebook produces the paper's Table 2 protocol for one more model: **QEvict at a 10% KV-cache budget**
on **DeepSeek-V2-Lite-Chat**, on the **11 English LongBench tasks of Table 2 other than NarrativeQA**,
plus the **Full-KV** reference row the table is read against.

| | |
|---|---|
| Tasks | Qasper, MultiFieldQA-en, HotpotQA, 2WikiMQA, MuSiQue, GovReport, QMSum, MultiNews, TREC, TriviaQA, SAMSum (all examples: 150 for MultiFieldQA-en, 200 for the rest) |
| Budget | `cache_budget = 0.10`, the paper's "10% KV-cache budget" (Table 2; Appendix B.3, rho_A = 0.10). The repo's `compression_ratio` field is `1 - cache_budget`, so this is *not* a 90% budget. |
| Method (Appendix B.5, C) | W = 8, R = 8, 5 sink tokens, **128 recent tokens** (LongBench, Appendix C), alpha = 0.70 of the historical byte budget to INT2 (`quant_budget_mode: bytes`), read ratio rho = 0.25, bidirectional F<->Q migration, INT2 promotion payload, first routing event right after prefill |
| Decoding (B.1) | greedy; same checkpoint, tokenizer, prompt template, truncation, generation limits and post-processing for both rows |
| Stack (H.6) | one A100 80GB; transformers 4.47.1; PyTorch 2.6.0 (+cu124); flash-attn 2.7.4.post1 (2.8.x wheels do not load against pip's torch 2.6.0, see section 2) |
| Prompts | THUDM `dataset2prompt` / `dataset2maxlen`, chat template on all but TREC/TriviaQA/SAMSum, middle truncation to **31,500 tokens** (DeepSeek-V2-Lite is a 32k model; THUDM's convention for 32k models). The Llama rows were not truncated (128k model). |

### What changes for DeepSeek-V2-Lite (MLA), and what to say in the paper
DeepSeek-V2 attention is Multi-head Latent Attention: per token and layer it caches one 512-wide latent `c` and one
64-wide RoPE key, shared by all 16 query heads. The port (`docs/deepseek_v2_lite/`) maps this onto the method as
**one KV head of width 576 (rep = 16) whose value is the key's latent dims**. Everything in Algorithms 1-2 then runs
unchanged: windows, cumulative fp32 scores, the byte-budgeted F/Q/E routing, write-once INT2 records, the rank-1
card, the per-head-share GQA union (here over 16 query heads), mean-log completion. Three consequences belong in
the paper's text:

1. **The value is not quantized separately.** Appendix J.1 quantizes keys per channel and values per token (KIVI).
   In MLA the value *is* the latent, so the port stores it **once, with per-channel grids** (the key rule), and the
   value centroid nu_w is the latent part of mu_w (same tokens, same mean), costing no extra bytes.
2. **Byte accounting is over the latent:** Full-KV is 1,152 B/token/layer (576 x bf16), not
   `2 * H_kv * d_h`. One INT2 window costs 2,376 B + an 878 B card.
3. **The logit scale is the model's own** (0.11472 = 192^-0.5 * mscale^2, YaRN), where Eq. 31 writes 1/sqrt(d_h).

### Guarantees this notebook enforces
* Every run is on one pinned commit and one pinned checkpoint revision; a Drive folder holds one commit.
* Before the long run: the DeepSeek CPU suite at that commit, a **GPU preflight** that checks each
  GPU-only path (FA2 prefill + score kernel, gate kernel, MLA decode kernel, compiled eviction) against its CPU
  reference on this GPU, and a smoke run on the single longest prompt.
* Every sidecar is verified before its predictions are kept: QEvict chunks must say `read_gate.verdict == "gated"`
  with a realised read fraction of about 0.25, and every method field must equal the settings below. Anything else is
  moved to `refused/` and the run stops.
* No example is skipped: `skip_oom` is off, and an out-of-memory error stops the run.

**Runtime is not measured** for this model on this stack. The notebook prints a projection after the first chunk.
Work is saved per (arm, task, chunk of 50 examples), so a disconnect loses at most one chunk; rerun all cells to resume.

## 0. Settings

In [ ]:
# ===================== Settings: edit here =====================
BRANCH         = "Clustered_int2_paused_fast_deepseek"
REPO_URL       = "https://github.com/shaswata965-coder/RecoverKv.git"
PIN_COMMIT     = "9bda7d291e8f50dc4c145351c6410d01b96a7742"   # every arm runs on this commit; quality rows do not compare across commits
MODEL_ID       = "deepseek-ai/DeepSeek-V2-Lite-Chat"
MODEL_REVISION = "85864749cd611b4353ce1decdb286193298f64c7"   # models/deepseek_v2/loader.PINNED_REVISIONS
MODEL_DIR      = "/content/models/DeepSeek-V2-Lite-Chat"      # the name must contain "chat": the chat-template gate reads it
DRIVE_ROOT     = "/content/drive/MyDrive/qevict_deepseek_v2_lite_longbench_b010"

ARMS_TO_RUN       = ["qevict_b010", "fullkv"]   # run in this order; Full-KV is the 100% reference row
CHUNK             = 50       # examples per resumable unit
RUN_CPU_TESTS     = True     # the DeepSeek CPU suite at the pinned commit (~6-10 min)
RUN_GPU_PREFLIGHT = True     # the GPU-only paths against their CPU references, on this GPU (~5 min)
RUN_SMOKE         = True     # 2 Qasper examples + the single longest prompt, both arms, before the long run
FLASH_ATTN_FROM_SOURCE = False   # only if no prebuilt wheel loads (slow: ~1-2 h)
ALLOW_NON_A100    = False    # an 80 GB H100 should work, but these kernels have only been compile-checked for sm80

# ===================== Fixed: these define the experiment =====================
TASKS = ["qasper", "multifieldqa_en", "hotpotqa", "2wikimqa", "musique",
         "gov_report", "qmsum", "multi_news", "trec", "triviaqa", "samsum"]   # Table 2 minus NarrativeQA
LABEL = {"qasper": "Qasper", "multifieldqa_en": "Mul.QA", "hotpotqa": "Hot.QA", "2wikimqa": "2Wi.QA",
         "musique": "Musique", "gov_report": "Gov.Re.", "qmsum": "QMSum", "multi_news": "M.News",
         "trec": "TREC", "triviaqa": "Tri.QA", "samsum": "SAMSum"}
EXPECT_N = {t: (150 if t == "multifieldqa_en" else 200) for t in TASKS}
METHOD = {"cache_budget": 0.10, "window_size": 8, "num_sink_tokens": 5, "local_window_size": 128,
          "quant_ratio": 0.70, "quant_budget_mode": "bytes", "quant_gate_ratio": 0.25,
          "first_eviction_step": 0}
POLICY = {"quant_promotion": "bidir", "quant_promote_source": "dequant"}
CARD_BITS = {"mu": 4, "v": 8, "t": 8, "vm": 4}
MAX_LENGTH = 31500
SEED = 42
ARMS = {
    "qevict_b010": {"config": "configs/longbench_deepseek_v2_lite_flash.yaml", "verdict": "gated",
                    "overrides": [f"cache.{k}={v}" for k, v in METHOD.items()],
                    "label": "QEvict", "kv": "10%"},
    "fullkv":      {"config": "configs/longbench_deepseek_v2_lite_full_cache.yaml", "verdict": "not-expected",
                    "overrides": [], "label": "Full-KV", "kv": "100%"},
}
assert set(ARMS_TO_RUN) <= set(ARMS), ARMS_TO_RUN

## 1. GPU, Drive, repository

In [ ]:
import subprocess, json, os, sys, shutil, time, math, zipfile, hashlib, statistics, csv
from pathlib import Path

q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                    "--format=csv,noheader,nounits"], capture_output=True, text=True)
if q.returncode != 0:
    raise RuntimeError("No GPU. Runtime -> Change runtime type -> A100 (80 GB).")
gpu_name, mem_mib, cc = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
GPU_MEM_GB = float(mem_mib) / 1024
print(f"GPU: {gpu_name}  {GPU_MEM_GB:.0f} GB  sm_{cc.replace('.', '')}")
if GPU_MEM_GB < 75:
    raise RuntimeError(
        f"{GPU_MEM_GB:.0f} GB is not enough. The bf16 weights are ~31 GB, and at a 31.5k-token prompt the "
        "remote DeepseekV2ForCausalLM computes fp32 logits for every prompt position (~13 GB, plus a bf16 copy). "
        "A paper row may not skip examples, so this needs an 80 GB card (the paper's A100 80GB).")
if cc != "8.0" and not ALLOW_NON_A100:
    raise RuntimeError(f"Expected an A100 (sm_80), got {gpu_name} (sm_{cc}). The kernels were compile-checked "
                       "for sm80 only. Set ALLOW_NON_A100 = True to run anyway (the GPU preflight still runs).")

In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = Path(DRIVE_ROOT)
for d in ("logs", "refused", "results", *ARMS):
    (ROOT / d).mkdir(parents=True, exist_ok=True)

def _secret(k):
    try:
        return userdata.get(k)
    except Exception:
        return None

REPO = Path("/content/RecoverKv")
gh = _secret("GITHUB_TOKEN")
url = REPO_URL.replace("https://", f"https://x-access-token:{gh}@") if gh else REPO_URL

def _git(*args, cwd=None):
    r = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True)
    if r.returncode != 0:  # never echo the tokenised URL
        raise RuntimeError("git " + args[0] + " failed:\n" + r.stderr.replace(gh or "\0", "***"))
    return r.stdout.strip()

if not REPO.exists():
    _git("clone", "--branch", BRANCH, url, str(REPO))
else:
    _git("fetch", "origin", BRANCH, cwd=REPO)
_git("checkout", "--force", PIN_COMMIT, cwd=REPO)
SHA = _git("rev-parse", "HEAD", cwd=REPO)
assert SHA.startswith(PIN_COMMIT[:10]), (SHA, PIN_COMMIT)
assert not _git("status", "--porcelain", "--untracked-files=no", cwd=REPO), "the clone has local changes"
for f in ("scripts/deepseek_gpu_preflight.py", "scripts/check_read_gate.py", ARMS["qevict_b010"]["config"],
          ARMS["fullkv"]["config"], "models/deepseek_v2/attention.py"):
    assert (REPO / f).exists(), f"{f} missing at {SHA[:10]}: wrong PIN_COMMIT"
print("commit", SHA)

# Quality rows do not compare across commits, so one Drive root holds one commit, one model, one method.
manifest = ROOT / "manifest.json"
want = {"sha": SHA, "branch": BRANCH, "model": MODEL_ID, "revision": MODEL_REVISION,
        "method": METHOD, "policy": POLICY, "card_bits": CARD_BITS, "max_length": MAX_LENGTH, "tasks": TASKS}
if manifest.exists():
    have = json.loads(manifest.read_text())
    diff = {k: (have.get(k), v) for k, v in want.items() if have.get(k) != v}
    if diff:
        raise RuntimeError(f"{ROOT} holds a different run: {diff}. Point DRIVE_ROOT at a new folder.")
else:
    manifest.write_text(json.dumps({**want, "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())},
                                   indent=2))

## 2. Install the pinned stack

Colab's kernel Python is newer than the pinned stack supports, so the stack goes into a separate **Python 3.12 venv**
(`/content/py312_env`) and every job runs with that interpreter (`PY`). The notebook kernel never imports torch.

* **torch 2.6.0** (+cu124): the build the recorded GPU runs used. On torch 2.7.1 the eviction's `torch.compile`
  fails at decode step 0 (`StarDep does not have an index`).
* **transformers 4.47.1**: the windowed cache refuses anything newer, and the DeepSeek remote code is pinned
  against it.
* **flash-attn** comes from a prebuilt wheel, installed with `--no-deps` under a constraints file, and kept only if
  its CUDA extension imports against the venv's torch. In practice that is **2.7.4.post1**. The paper lists 2.8.3,
  but the 2.8.x `cu12torch2.6cxx11abiFALSE` wheels are built with the new C++ ABI and fail against pip's torch 2.6.0
  (`undefined symbol: _ZN3c105ErrorC2ENS_14SourceLocationENSt7__cxx11...`). The FA2 forward used here
  (`flash_attn_func` with `return_attn_probs`) is the same API in both.
* The check at the end runs FA2 at DeepSeek's decompressed head width (192) and checks that `softmax_lse` has the
  shape the score kernel reads.

In [ ]:
import importlib.util

VENV = Path("/content/py312_env")
PY = str(VENV / "bin" / "python")
PINS = {"torch": "2.6.0", "torchvision": "0.21.0", "torchaudio": "2.6.0", "triton": "3.2.0",
        "transformers": "4.47.1"}
STACK_ID = json.dumps({"python": "3.12", "cuda": "cu124", **PINS}, sort_keys=True)
MARK = VENV / ".recoverkv_stack"

_DROP = ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "CPATH", "C_INCLUDE_PATH", "CPLUS_INCLUDE_PATH",
         "PIP_CONSTRAINT", "MPLBACKEND")
VENV_ENV = {k: v for k, v in os.environ.items() if k not in _DROP}
VENV_ENV.update(VIRTUAL_ENV=str(VENV), PATH=f"{VENV / 'bin'}:{os.environ.get('PATH', '')}", MPLBACKEND="Agg")

def sh(*cmd, env=None):
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd)[:300])
    subprocess.run(cmd, check=True, env=VENV_ENV if env is None else env)

def vpip(*a):
    sh(PY, "-m", "pip", "install", "-q", "--disable-pip-version-check", *a)

def venv_versions():
    code = ("import json, importlib.metadata as m\nout = {}\n"
            "for d in ('torch','torchvision','torchaudio','triton','transformers','flash_attn','datasets','numpy'):\n"
            "    try: out[d] = m.version(d)\n    except m.PackageNotFoundError: out[d] = None\n"
            "print(json.dumps(out))")
    return json.loads(subprocess.check_output([PY, "-c", code], env=VENV_ENV, text=True))

def check_pins():
    v = venv_versions()
    bad = {k: v.get(k) for k, want in PINS.items() if (v.get(k) or "").split("+")[0] != want}
    if bad:
        raise RuntimeError(f"the venv drifted from the pinned stack: {bad}, want {PINS}. Delete {VENV} and rerun.")
    return v

if not (MARK.exists() and MARK.read_text() == STACK_ID and Path(PY).exists()):
    if VENV.exists():
        shutil.rmtree(VENV)
    has312 = shutil.which("python3.12") and subprocess.run(
        ["python3.12", "-c", "import ensurepip, venv"], capture_output=True).returncode == 0
    if not has312 or FLASH_ATTN_FROM_SOURCE:
        apt_env = dict(os.environ, DEBIAN_FRONTEND="noninteractive")
        sh("apt-get", "update", "-qq", env=apt_env)
        sh("apt-get", "install", "-y", "-qq", "python3.12", "python3.12-venv", "python3.12-dev", env=apt_env)
    sh("python3.12", "-m", "venv", VENV, env=os.environ)
    constraints = VENV / "constraints.txt"
    constraints.write_text("".join(f"{k}=={v}\n" for k, v in PINS.items()))
    vpip("--upgrade", "pip")
    vpip("torch==2.6.0", "torchvision==0.21.0", "torchaudio==2.6.0",
         "--index-url", "https://download.pytorch.org/whl/cu124")
    vpip("-c", constraints,
         "transformers==4.47.1", "accelerate>=0.25,<2", "datasets>=2.16,<3", "huggingface_hub>=0.24,<1.0",
         "sentencepiece", "protobuf", "einops>=0.7", "scipy", "pyyaml", "tqdm", "pandas",
         "rouge", "fuzzywuzzy", "python-Levenshtein", "jieba", "pytest", "pyflakes")
    check_pins()
    tv, abi, py = subprocess.check_output([PY, "-c",
        "import torch, sys; print(torch.__version__.split('+')[0]); "
        "print(int(torch._C._GLIBCXX_USE_CXX11_ABI)); "
        "print(f'cp{sys.version_info.major}{sys.version_info.minor}')"], env=VENV_ENV, text=True).split()
    tmm = ".".join(tv.split(".")[:2])

    def _fa_loads():
        r = subprocess.run([PY, "-c", "import torch, flash_attn_2_cuda, flash_attn; print(flash_attn.__version__)"],
                           env=VENV_ENV, capture_output=True, text=True)
        return r.returncode == 0, (r.stdout + r.stderr).strip().splitlines()[-1:]

    wheel = None
    for v in ("2.7.4.post1", "2.8.3", "2.8.2"):
        u = (f"https://github.com/Dao-AILab/flash-attention/releases/download/v{v}/"
             f"flash_attn-{v}+cu12torch{tmm}cxx11abi{'TRUE' if abi == '1' else 'FALSE'}-{py}-{py}-linux_x86_64.whl")
        try:
            vpip("--no-deps", "-c", constraints, u)
        except subprocess.CalledProcessError:
            print("   no such wheel")
            continue
        ok, why = _fa_loads()
        if ok:
            wheel = u
            break
        print("   installs but does not load against torch", tv, "->", *why)
        sh(PY, "-m", "pip", "uninstall", "-y", "-q", "flash-attn")
    if wheel:
        print("flash-attn wheel:", wheel.rsplit("/", 1)[1])
    elif FLASH_ATTN_FROM_SOURCE:
        vpip("-c", constraints, "ninja", "packaging", "wheel", "psutil", "setuptools")
        sh(PY, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-c", constraints,
           "flash-attn==2.7.4.post1", env=dict(VENV_ENV, MAX_JOBS="8", FLASH_ATTENTION_FORCE_BUILD="TRUE"))
        ok, why = _fa_loads()
        if not ok:
            raise RuntimeError(f"flash-attn built from source does not load: {why}")
    else:
        raise RuntimeError(f"No prebuilt flash-attn wheel loads against torch {tv} / {py} / cxx11abi={abi}. "
                           "Set FLASH_ATTN_FROM_SOURCE = True (about 1-2 h) and rerun this cell.")
print("venv:", check_pins())

check = r"""
import torch
print("torch", torch.__version__, "| cuda", torch.version.cuda)
assert torch.__version__.startswith("2.6."), "torch must be 2.6.x: the eviction does not compile on 2.7.1"
assert torch.cuda.is_available(), "CUDA is not available inside the venv"
import triton, transformers, flash_attn, datasets
print("triton", triton.__version__, "| transformers", transformers.__version__,
      "| flash_attn", flash_attn.__version__, "| datasets", datasets.__version__)
from flash_attn import flash_attn_func
# DeepSeek's prefill: 16 heads of the decompressed 192-wide head, V padded to 192, L returned for the score kernel.
x = torch.randn(1, 512, 16, 192, device="cuda", dtype=torch.bfloat16)
out, lse, _ = flash_attn_func(x, x, x, 0.0, softmax_scale=0.11472, causal=True, return_attn_probs=True)
torch.cuda.synchronize()
assert out.shape == x.shape and tuple(lse.shape) == (1, 16, 512) and lse.dtype == torch.float32, (out.shape, lse.shape)
f = torch.compile(lambda t: (t * 2).sin().sum(-1))     # the eviction is compiled-or-raise
f(torch.randn(64, 64, device="cuda")); torch.cuda.synchronize()
from utils.cache_factory import assert_transformers_version_supported
assert_transformers_version_supported()
print("environment ok")
"""
r = subprocess.run([PY, "-c", check], cwd=REPO, env=VENV_ENV, capture_output=True, text=True)
print(r.stdout, r.stderr[-4000:] if r.returncode else "")
if r.returncode != 0:
    raise RuntimeError("environment check failed inside the venv; see the output above")
MARK.write_text(STACK_ID)

## 3. Model and data

The checkpoint is downloaded at the pinned revision. Its remote code must hash to the copy vendored in the repo
(`models/deepseek_v2/upstream/`), which is what the owned MLA forward was checked against. Jobs run with
`HF_HUB_OFFLINE=1`, so nothing can be fetched or re-resolved mid-run.

LongBench comes from `THUDM/LongBench`'s `data.zip`. Each task is cut into chunk files of `CHUNK` examples, copied
line for line, so a job reads exactly the examples it was given and the merged predictions are in the original order.

In [ ]:
from huggingface_hub import snapshot_download, hf_hub_download
HF_TOKEN = _secret("HF_TOKEN")      # optional: the checkpoint and the dataset are public
snapshot_download(MODEL_ID, revision=MODEL_REVISION, local_dir=MODEL_DIR, token=HF_TOKEN,
                  allow_patterns=["*.json", "*.safetensors", "*.py", "tokenizer*"])
PINNED_FILES = {
    "modeling_deepseek.py": "7d8e5221095286eea991137760893fd7ba52727c0b4ebf48ec09e8bc56b45b9c",
    "configuration_deepseek.py": "6de5e7445ab490a16482027b131d5bd6a003da9b58a182a11432115fac84ce59",
    "config.json": "f346286b0f1c8b044252fd54cb4fa78b9fab6472a6e8bebb9edfe03d414ea03d",
}
for f, h in PINNED_FILES.items():
    got = hashlib.sha256((Path(MODEL_DIR) / f).read_bytes()).hexdigest()
    if got != h:
        raise RuntimeError(f"{f} hashes to {got[:16]}, not the pinned {h[:16]}: wrong checkpoint revision")
    vend = hashlib.sha256((REPO / "models/deepseek_v2/upstream" / f).read_bytes()).hexdigest()
    assert vend == h, f"the repo's vendored {f} moved"
assert "chat" in Path(MODEL_DIR).name.lower(), "MODEL_DIR's name must contain 'chat'"
print("checkpoint OK:", MODEL_ID, MODEL_REVISION[:12])

LB_DIR = Path("/content/longbench")
LB_DIR.mkdir(parents=True, exist_ok=True)
need = [t for t in TASKS if not (LB_DIR / f"{t}.jsonl").exists()]
if need:
    zp = hf_hub_download("THUDM/LongBench", "data.zip", repo_type="dataset", token=HF_TOKEN)
    with zipfile.ZipFile(zp) as z:
        names = z.namelist()
        for t in need:
            member = next(n for n in names if n == f"{t}.jsonl" or n.endswith(f"/{t}.jsonl"))
            with z.open(member) as src, open(LB_DIR / f"{t}.jsonl", "wb") as dst:
                shutil.copyfileobj(src, dst)
LINES, IDS = {}, {}
for t in TASKS:
    LINES[t] = [l for l in (LB_DIR / f"{t}.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
    IDS[t] = [json.loads(l)["_id"] for l in LINES[t]]
    assert len(IDS[t]) == EXPECT_N[t], f"{t}: {len(IDS[t])} examples, expected {EXPECT_N[t]}"
    assert len(set(IDS[t])) == len(IDS[t]), f"{t}: duplicate _id"
NCHUNKS = {t: -(-EXPECT_N[t] // CHUNK) for t in TASKS}
CHUNKS = Path(f"/content/chunks_{CHUNK}")
for t in TASKS:
    for k in range(NCHUNKS[t]):
        p = CHUNKS / f"{k:03d}" / f"{t}.jsonl"
        p.parent.mkdir(parents=True, exist_ok=True)
        p.write_text("\n".join(LINES[t][k * CHUNK:(k + 1) * CHUNK]) + "\n", encoding="utf-8")
print({t: len(IDS[t]) for t in TASKS}, f"in chunks of {CHUNK}")

ENV = dict(VENV_ENV, PYTHONHASHSEED="0", CUBLAS_WORKSPACE_CONFIG=":4096:8", TOKENIZERS_PARALLELISM="false",
           HF_HUB_OFFLINE="1", HF_DATASETS_CACHE="/content/hf_datasets_cache", PYTHONUNBUFFERED="1")

## 4. Readiness (refuses to start if anything is off)

1. The read gate is wired: `fused_gate` has non-test callers.
2. Every config in the repo is at the operating point (`scripts/check_operating_point.py`). The 10% budget is
   applied below as an override, which is how the paper's Llama 10% rows were launched
   (`scripts/run_longbench16_b010_b005.sh`).
3. The DeepSeek CPU suite at the pinned commit (`tests/test_deepseek_mla.py`, on this stack's torch 2.6): the owned
   forward against upstream, RoPE bit-identity, aliasing changes no eviction decision, the latent byte pricing, the
   compiled eviction over the latent, the runners' cache construction. The kernels' Triton-interpreter tests are not
   run here: Triton 3.2's interpreter cannot execute any of this repo's kernels (it passes scalar kernel arguments as
   one-element arrays), shipped or MLA; they pass under Triton 3.9's interpreter, and step 4 checks them compiled.
4. **The GPU preflight** (`scripts/deepseek_gpu_preflight.py`): on a small random model with this checkpoint's own
   attention geometry, on this GPU, it checks the owned forward (FA2 prefill) against upstream, the Triton score kernel
   against eager scores, the fused decode against the materialize path in bf16 and in fp32, the gate verdict at
   rho = 0.25, and that every eviction was compiled.

In [ ]:
probe = r"""
import json, subprocess
callers = subprocess.run("grep -rn 'fused_gate' --include='*.py' . | grep -v '^./tests/'",
                         shell=True, capture_output=True, text=True).stdout.splitlines()
print(json.dumps([c for c in callers if "def fused_gate" not in c]))
"""
callers = json.loads(subprocess.check_output([PY, "-c", probe], cwd=REPO, env=ENV, text=True))
op = subprocess.run([PY, "scripts/check_operating_point.py"], cwd=REPO, env=ENV, capture_output=True, text=True)
READY = {"gate_wired": bool(callers), "operating_point": op.returncode == 0}
print(", ".join(f"{k}={v}" for k, v in READY.items()))
if not all(READY.values()):
    print(op.stdout[-2000:])
    raise RuntimeError("not ready: " + ", ".join(k for k, v in READY.items() if not v))

# A pass is remembered on Drive (this root holds one commit), keyed by the GPU it ran on, so a resumed
# session does not redo ~15 minutes of checks. Delete the files to force them.
CPU_OK, GPU_OK = ROOT / "results" / "cpu_tests_ok.json", ROOT / "results" / "gpu_preflight.json"
if RUN_CPU_TESTS and CPU_OK.exists():
    print("CPU suites already passed at this commit:", json.loads(CPU_OK.read_text())["utc"])
elif RUN_CPU_TESTS:
    t = subprocess.run([PY, "-m", "pytest", "-q", "-p", "no:cacheprovider",
                        "tests/test_deepseek_mla.py"],
                       # The tests build CPU tensors themselves. The GPU stays visible on purpose: Triton 3.2's
                       # autotuner asks the active GPU driver for a benchmarker at import, so hiding the GPU would
                       # make the repo unimportable. ATEN_CPU_CAPABILITY: scalar CPU codegen for the CPU-side tests.
                       cwd=REPO, env=dict(ENV, ATEN_CPU_CAPABILITY="default"),
                       capture_output=True, text=True)
    print(t.stdout[-1500:] if t.returncode == 0 else t.stdout[-8000:] + t.stderr[-3000:])
    if t.returncode != 0:
        raise RuntimeError("the DeepSeek CPU suite failed at this commit; see the output above")
    CPU_OK.write_text(json.dumps({"sha": SHA, "utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
                                  "summary": t.stdout.strip().splitlines()[-1]}))

prev = json.loads(GPU_OK.read_text()) if GPU_OK.exists() else {}
if RUN_GPU_PREFLIGHT and prev.get("ok") and prev.get("gpu_name") == gpu_name:
    print(f"GPU preflight already passed on {gpu_name} at this commit")
elif RUN_GPU_PREFLIGHT:
    t = subprocess.run([PY, "scripts/deepseek_gpu_preflight.py", "--model-dir", MODEL_DIR,
                        "--budget", str(METHOD["cache_budget"]), "--gate", str(METHOD["quant_gate_ratio"])],
                       cwd=REPO, env=ENV, capture_output=True, text=True)
    lines = t.stdout.strip().splitlines()
    print("\n".join(l for l in lines if l.startswith(("PASS", "FAIL"))) or t.stdout[-3000:])
    if t.returncode != 0 or not lines:
        print(t.stderr[-6000:])
        raise RuntimeError("GPU preflight failed: a GPU-only path disagrees with its CPU reference on this GPU. "
                           "Do not run the benchmark on this stack; see the output above.")
    GPU_OK.write_text(json.dumps({**json.loads(lines[-1]), "gpu_name": gpu_name, "sha": SHA}))
print("ready")

## 5. Budget geometry, before any GPU time is spent

Every example's prompt is built exactly as `LongBenchRunner._predict` builds it (template, middle truncation to
31,500 tokens, chat template gate, BOS handling), with this checkpoint's tokenizer, and resolved at the 10% budget by
the cache's own resolver. Per task it reports how many examples get an INT2 tier at all (when sink + recent tokens
exceed the budget the cache keeps them anyway, holds no evictable window, and the gate has nothing to select), the
tier sizes K_F and K_Q, and the KV memory two ways:

* **nominal**: the active allocation the budget resolver spends, over Full-KV (Appendix B.4: "the nominal active
  tier allocation used to determine K_F and K_Q"). This is how a 10% row is configured.
* **physical**: Appendix B.4's Eq. 9 with this cache's allocations: the native-precision buffer as allocated
  (sink + recent + K_F windows + two windows of slack), the INT2 slot table `N_slot = K_F + K_Q + 8` (Eq. 10) at the
  full per-window cost (codes, grids, card), positions (int64) and the slot ledger (id int64 + active flag).

Full-KV is the latent cache, 1,152 B/token/layer, over prompt + the task's generation limit (the horizon the budget
is sized against). Both are per layer per sequence; every layer resolves identically.

In [ ]:
geom_src = r"""
import json, sys, statistics, warnings
warnings.simplefilter("ignore")
import torch
from transformers import AutoTokenizer
from utils.prompting import encode_prompt
from models.deepseek_v2.geometry import MLAGeometry
from models.deepseek_v2.upstream.configuration_deepseek import DeepseekV2Config
from modules.windowed_cache.config import WindowedCacheConfig
from modules.evaluation.longbench_runner import LongBenchRunner
from modules.quant.slots import n_slots_for
model_dir, lb_dir, max_length, method, tasks = sys.argv[1], sys.argv[2], int(sys.argv[3]), json.loads(sys.argv[4]), json.loads(sys.argv[5])
tok = AutoTokenizer.from_pretrained(model_dir)
d = json.load(open(f"{model_dir}/config.json"))
for k in ("architectures", "auto_map", "model_type", "transformers_version", "torch_dtype"):
    d.pop(k, None)
mc = MLAGeometry.from_config(DeepseekV2Config(**d)).cache_config()
cfg = WindowedCacheConfig(**{k: method[k] for k in ("cache_budget", "window_size", "num_sink_tokens",
                             "local_window_size", "quant_ratio", "quant_budget_mode", "quant_gate_ratio",
                             "first_eviction_step")})
d2p = json.load(open("data/longbench_configs/dataset2prompt.json"))
d2m = json.load(open("data/longbench_configs/dataset2maxlen.json"))
out = {}
for t in tasks:
    rows = []
    for line in open(f"{lb_dir}/{t}.jsonl", encoding="utf-8"):
        if not line.strip():
            continue
        ex = json.loads(line)
        prompt = d2p[t].format(context=ex["context"], input=ex.get("input", ""))
        ids = tok(prompt, truncation=False, return_tensors="pt").input_ids[0]
        if len(ids) > max_length:
            half = max_length // 2
            prompt = tok.decode(ids[:half], skip_special_tokens=True) + tok.decode(ids[-half:], skip_special_tokens=True)
        if LongBenchRunner._should_apply_chat_template(model_dir, t):
            prompt = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False,
                                             add_generation_prompt=True)
        enc = encode_prompt(tok, prompt, truncation=False, return_tensors="pt").input_ids
        n, g = int(enc.shape[-1]), int(d2m[t])
        assert not (enc[0, 0] == tok.bos_token_id and enc[0, 1] == tok.bos_token_id), "duplicate BOS"
        r = cfg.resolve(n, mc, torch.bfloat16, g)
        W, bpt = r.window_size, r.bytes_per_token
        full = (n + g) * bpt
        slots = n_slots_for(r.top_k_fp, r.N_q)
        fp_alloc = r.num_sink_tokens + r.local_tokens + r.top_k_fp * W + 2 * W
        physical = (fp_alloc * bpt + slots * r.bytes_per_q_window + fp_alloc * 8 + slots * W * 8 + slots * 9)
        rows.append({"prompt": n, "gen": g, "K_F": r.top_k_fp, "K_Q": r.N_q,
                     "nominal": r.retained_bytes / full, "physical": physical / full,
                     "over_context": n + g > 32768})
    out[t] = {"n": len(rows),
              "prompt_min": min(r["prompt"] for r in rows),
              "prompt_median": statistics.median(r["prompt"] for r in rows),
              "prompt_max": max(r["prompt"] for r in rows),
              "no_int2_tier": sum(r["K_Q"] == 0 for r in rows),
              "over_32k": sum(r["over_context"] for r in rows),
              "K_F_mean": statistics.mean(r["K_F"] for r in rows),
              "K_Q_mean": statistics.mean(r["K_Q"] for r in rows),
              "nominal_mean": statistics.mean(r["nominal"] for r in rows),
              "physical_mean": statistics.mean(r["physical"] for r in rows),
              "longest_prompt": max(range(len(rows)), key=lambda i: rows[i]["prompt"])}
print(json.dumps(out))
"""
GEOM_FILE = ROOT / "results" / "geometry_b010.json"
if not GEOM_FILE.exists():
    g = subprocess.run([PY, "-c", geom_src, MODEL_DIR, str(LB_DIR), str(MAX_LENGTH), json.dumps(METHOD),
                        json.dumps(TASKS)], cwd=REPO, env=ENV, capture_output=True, text=True)
    if g.returncode != 0:
        raise RuntimeError(g.stderr[-4000:])
    GEOM_FILE.write_text(g.stdout.strip().splitlines()[-1])
GEOM = json.loads(GEOM_FILE.read_text())
print(f"{'task':18s}{'n':>5s}{'prompt min/med/max':>22s}{'no INT2':>9s}{'K_F':>7s}{'K_Q':>8s}{'nominal':>9s}{'physical':>10s}")
for t in TASKS:
    s = GEOM[t]
    print(f"{t:18s}{s['n']:5d}{s['prompt_min']:>8d}/{int(s['prompt_median']):>6d}/{s['prompt_max']:>6d}"
          f"{s['no_int2_tier']:>9d}{s['K_F_mean']:>7.1f}{s['K_Q_mean']:>8.1f}"
          f"{100 * s['nominal_mean']:>8.2f}%{100 * s['physical_mean']:>9.2f}%")
    assert s["over_32k"] == 0, f"{t}: prompt + generation exceeds the 32k context"
allw = [(GEOM[t]["n"], GEOM[t]) for t in TASKS]
tot = sum(n for n, _ in allw)
print(f"\nall {tot} examples: no INT2 tier on {sum(s['no_int2_tier'] for _, s in allw)}; "
      f"KV memory nominal {100 * sum(n * s['nominal_mean'] for n, s in allw) / tot:.2f}%, "
      f"physical {100 * sum(n * s['physical_mean'] for n, s in allw) / tot:.2f}% (example-weighted means)")

## 6. Jobs, verification and merging

One job = one arm x one chunk index k, running every pending task's chunk k in one process (one model load). The
runner writes a task's predictions and then its sidecar, so the sidecar is the done-marker. Before a chunk is kept,
its sidecar must match the arm exactly (verdict, method fields, card widths, migration policy, model, revision,
commit, stack, truncation, example count) and its predictions must be the chunk's examples in order. A task is merged
into `<arm>/final/<task>.jsonl` only when all its chunks are verified; the merged sidecar keeps every chunk's verdict.

In [ ]:
LOGS, RUNS = ROOT / "logs", Path("/content/runs")

def run(cmd, log_name, env=None):
    cmd = [str(c) for c in cmd]
    log = LOGS / f"{log_name}.log"
    print("$", " ".join(cmd)[:400])
    t0, tail = time.time(), []
    keep = ("examples done", "read gate", "examples,", "OOM", "Error", "error:", "NOT-GATED", "Traceback",
            "eviction path", "score path")
    with open(log, "w") as fh:
        p = subprocess.Popen(cmd, cwd=REPO, env=env or ENV, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout:
            fh.write(line)
            tail = (tail + [line])[-80:]
            if any(k in line for k in keep):
                print("   ", line.rstrip()[:220])
        rc = p.wait()
    if rc != 0:
        print("".join(tail))
        raise RuntimeError(f"{log_name} failed (rc={rc}). Full log: {log}")
    print(f"    finished in {(time.time() - t0) / 60:.1f} min")

def _same(g, v):
    if isinstance(v, (int, float)) and not isinstance(v, bool):
        return g is not None and not isinstance(g, bool) and abs(float(g) - float(v)) <= 1e-9 * max(1.0, abs(float(v)))
    return g == v

def verify(meta, arm, ids):
    """Everything a chunk's sidecar must say for its predictions to belong to this arm."""
    A, errs = ARMS[arm], []
    rg = meta.get("read_gate") or {}
    if rg.get("verdict") != A["verdict"]:
        errs.append(f"read_gate verdict {rg.get('verdict')!r}, need {A['verdict']!r}")
    expect = {"attn_implementation": "flash_attention_2", "dtype": "bfloat16", "max_length": MAX_LENGTH,
              "model_name": MODEL_DIR, "model_revision": MODEL_REVISION, "commit_sha": SHA,
              "transformers_version": "4.47.1", "num_examples": len(ids), "seed": SEED}
    if arm == "qevict_b010":
        rho = METHOD["quant_gate_ratio"]
        rf = rg.get("read_fraction")
        if rf is None or not (rho - 1e-6 <= rf <= rho + 0.15):
            errs.append(f"realised read fraction {rf} for ratio {rho}")
        if not rg.get("fired") or rg.get("gated") != rg.get("fired"):
            errs.append(f"fired={rg.get('fired')} gated={rg.get('gated')}")
        expect.update({k: v for k, v in METHOD.items()})
        expect.update(POLICY)
        expect.update({"cache_type": "windowed", "cache_backend_package": "flash_attn",
                       "quant_card_bits": CARD_BITS, "rerotate_on_evict": False})
        geo = meta.get("resolved_geometry_first_example") or {}
        if geo.get("quant_budget_mode") != "bytes":
            errs.append(f"resolved geometry {geo}")
    else:
        expect.update({"cache_type": "full_cache"})
    for k, v in expect.items():
        if not _same(meta.get(k), v):
            errs.append(f"{k} = {meta.get(k)!r}, expected {v!r}")
    if not str(meta.get("torch_version", "")).startswith("2.6."):
        errs.append(f"torch_version {meta.get('torch_version')!r}")
    return errs

def _ids_of(jsonl):
    return [json.loads(l)["_id"] for l in Path(jsonl).read_text(encoding="utf-8").splitlines() if l.strip()]

def _refuse(files, tag, errs):
    dst = ROOT / "refused" / tag
    dst.mkdir(parents=True, exist_ok=True)
    for f in files:
        if Path(f).exists():
            shutil.move(str(f), dst / Path(f).name)
    raise RuntimeError(f"REFUSED {tag}:\n  " + "\n  ".join(errs) + f"\n  files moved to {dst}")

def chunk_ids(t, k):
    return IDS[t][k * CHUNK:(k + 1) * CHUNK]

def stored(arm, t, k):
    d = ROOT / arm / "chunks"
    return d / f"{t}.{k:03d}.jsonl", d / f"{t}.{k:03d}.meta.json"

def chunk_done(arm, t, k):
    pf, mf = stored(arm, t, k)
    if not (pf.exists() and mf.exists()):
        return False
    errs = verify(json.loads(mf.read_text()), arm, chunk_ids(t, k))
    if _ids_of(pf) != chunk_ids(t, k):
        errs.append("predictions are not this chunk's examples in order")
    if errs:
        raise RuntimeError(f"STALE {arm}/{t}/chunk {k} on Drive does not match this run:\n  " + "\n  ".join(errs)
                           + "\n  Move it away to rerun it.")
    return True

def overrides(arm, out_dir, tasks):
    return ["--override", f"model.name={MODEL_DIR}", f"model.revision={MODEL_REVISION}",
            f"longbench.datasets=[{','.join(tasks)}]", f"longbench.output_dir={out_dir}",
            "longbench.num_samples=max", f"longbench.max_length={MAX_LENGTH}", "longbench.resume=false",
            "longbench.skip_oom=false", f"run.seed={SEED}", *ARMS[arm]["overrides"]]

def run_chunk(arm, k):
    todo = [t for t in TASKS if k < NCHUNKS[t] and not chunk_done(arm, t, k)]
    if not todo:
        return False
    local = RUNS / arm / f"chunk{k:03d}"
    shutil.rmtree(local, ignore_errors=True)
    local.mkdir(parents=True)
    failure = None
    try:
        run([PY, "main.py", "--config", ARMS[arm]["config"], *overrides(arm, local, todo)],
            f"{arm}_chunk{k:03d}", env=dict(ENV, LONGBENCH_LOCAL_DIR=str(CHUNKS / f"{k:03d}")))
    except RuntimeError as e:
        failure = e
    for t in todo:
        pf, mf = local / f"{t}.jsonl", local / f"{t}.meta.json"
        if not (pf.exists() and mf.exists()):
            continue
        errs = verify(json.loads(mf.read_text()), arm, chunk_ids(t, k))
        if _ids_of(pf) != chunk_ids(t, k):
            errs.append("predictions are not this chunk's examples in order")
        if errs:
            _refuse([pf, mf], f"{arm}_{t}_chunk{k:03d}", errs)
        dp, dm = stored(arm, t, k)
        dp.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(pf, dp)
        shutil.copy(mf, dm)
    if failure is not None:
        raise failure
    return True

def merge(arm, t):
    fin = ROOT / arm / "final"
    fin.mkdir(parents=True, exist_ok=True)
    if (fin / f"{t}.meta.json").exists():
        return True
    if not all(chunk_done(arm, t, k) for k in range(NCHUNKS[t])):
        return False
    lines, chunks = [], []
    for k in range(NCHUNKS[t]):
        pf, mf = stored(arm, t, k)
        lines += [l for l in pf.read_text(encoding="utf-8").splitlines() if l.strip()]
        m = json.loads(mf.read_text())
        chunks.append({"chunk": k, "num_examples": m["num_examples"], "read_gate": m["read_gate"],
                       "examples_per_second": m.get("examples_per_second"),
                       "run_started_utc": m.get("run_started_utc"), "run_finished_utc": m.get("run_finished_utc"),
                       "resolved_geometry_first_example": m.get("resolved_geometry_first_example")})
    ids = [json.loads(l)["_id"] for l in lines]
    assert ids == IDS[t], f"{arm}/{t}: merged predictions are not the task's examples in order"
    meta = json.loads(stored(arm, t, 0)[1].read_text())
    for k in ("read_gate", "examples_per_second", "run_started_utc", "run_finished_utc",
              "resolved_geometry_first_example"):
        meta.pop(k, None)
    verdicts = sorted({c["read_gate"]["verdict"] for c in chunks})
    rfs = [c["read_gate"].get("read_fraction") for c in chunks]
    # One verdict for the task only if every chunk says the same thing. Each chunk's own counters stay in
    # `chunks` (they are cumulative over the tasks its process ran, so they are not summed here).
    meta["read_gate"] = {"verdict": verdicts[0] if len(verdicts) == 1 else "NOT-GATED-partial",
                         "expected": chunks[0]["read_gate"].get("expected"),
                         "read_fraction": (statistics.mean(rfs) if all(r is not None for r in rfs) else None),
                         "fired": None, "gated": None, "aggregated_over_chunks": len(chunks)}
    meta.update({"num_examples": len(ids), "chunk_size": CHUNK, "chunks": chunks,
                 "read_gate_verdicts": verdicts, "arm": arm, "dataset": t})
    (fin / f"{t}.jsonl").write_text("\n".join(lines) + "\n", encoding="utf-8")
    (fin / f"{t}.meta.json").write_text(json.dumps(meta, indent=2, default=str))
    return True

def status():
    eps = []
    print(f"\n{'arm':14s}" + "".join(f"{LABEL[t]:>9s}" for t in TASKS))
    remaining = 0.0
    for arm in ARMS_TO_RUN:
        cells = []
        for t in TASKS:
            done = sum(stored(arm, t, k)[1].exists() for k in range(NCHUNKS[t]))
            cells.append(f"{done}/{NCHUNKS[t]}")
            for k in range(NCHUNKS[t]):
                mf = stored(arm, t, k)[1]
                if mf.exists():
                    e = json.loads(mf.read_text()).get("examples_per_second")
                    if e:
                        eps.append((arm, t, e))
        print(f"{arm:14s}" + "".join(f"{c:>9s}" for c in cells))
    if eps:
        rate = {}
        for arm, t, e in eps:
            rate.setdefault((arm, t), []).append(e)
        for arm in ARMS_TO_RUN:
            for t in TASKS:
                todo = sum(len(chunk_ids(t, k)) for k in range(NCHUNKS[t]) if not stored(arm, t, k)[1].exists())
                r = rate.get((arm, t)) or [e for (a2, t2), v in rate.items() if t2 == t for e in v]
                if todo and r:
                    remaining += todo / statistics.mean(r)
        print(f"\nprojected remaining (tasks with a measured rate only): {remaining / 3600:.1f} h")

## 7. Smoke test on real prompts (not part of the results)

Both arms on the **single longest prompt** of the 11 tasks (the memory worst case, at its task's generation limit), and
QEvict on the first two Qasper examples. Each sidecar must verify. Outputs go to `/content/runs/smoke`, not to the
results.

In [ ]:
if RUN_SMOKE and not (ROOT / "results" / "smoke_ok.json").exists():
    longest = max(TASKS, key=lambda t: GEOM[t]["prompt_max"])
    li = GEOM[longest]["longest_prompt"]
    sets = {"qasper": [0, 1]} if longest != "qasper" else {}
    sets.setdefault(longest, []).append(li)
    sdir = Path("/content/chunks_smoke")
    shutil.rmtree(sdir, ignore_errors=True)
    sdir.mkdir(parents=True)
    for t, idx in sets.items():
        (sdir / f"{t}.jsonl").write_text("\n".join(LINES[t][i] for i in idx) + "\n", encoding="utf-8")
    report = {"longest_task": longest, "longest_prompt_tokens": GEOM[longest]["prompt_max"]}
    for arm in ARMS_TO_RUN:
        tasks = list(sets) if arm == "qevict_b010" else [longest]
        local = RUNS / "smoke" / arm
        shutil.rmtree(local, ignore_errors=True)
        run([PY, "main.py", "--config", ARMS[arm]["config"], *overrides(arm, local, tasks)],
            f"smoke_{arm}", env=dict(ENV, LONGBENCH_LOCAL_DIR=str(sdir)))
        for t in tasks:
            meta = json.loads((local / f"{t}.meta.json").read_text())
            errs = verify(meta, arm, [IDS[t][i] for i in sets[t]])
            if errs:
                raise RuntimeError(f"smoke {arm}/{t} does not verify:\n  " + "\n  ".join(errs))
            report[f"{arm}/{t}"] = {"read_gate": meta["read_gate"],
                                    "examples_per_second": meta["examples_per_second"]}
    (ROOT / "results" / "smoke_ok.json").write_text(json.dumps(report, indent=2))
    print(json.dumps(report, indent=2))
print("smoke ok")

## 8. Run

Arms run in `ARMS_TO_RUN` order, chunk by chunk. Rerunning this cell resumes: verified chunks on Drive are skipped,
a partial chunk is redone from scratch.

In [ ]:
for arm in ARMS_TO_RUN:
    for k in range(max(NCHUNKS.values())):
        if run_chunk(arm, k):
            status()
    for t in TASKS:
        merge(arm, t)
status()
finals = [str(ROOT / arm / "final") for arm in ARMS_TO_RUN
          if all((ROOT / arm / "final" / f"{t}.meta.json").exists() for t in TASKS)]
if finals:
    r = subprocess.run([PY, "scripts/check_read_gate.py", *finals], cwd=REPO, env=ENV, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        raise RuntimeError("a merged sidecar does not prove what ran; see above")

## 9. Scores and the table

Scored with the repo's LongBench scorer (THUDM's metrics and post-processing). The average is over the **11 tasks
run here**; the paper's Table 2 averages 12 (with NarrativeQA), so the two averages do not compare. DeepSeek rows do
not compare with the Llama or Mistral rows either: different model, different cache, different truncation.

In [ ]:
SCORES = {}
for arm in ARMS_TO_RUN:
    fin = ROOT / arm / "final"
    if not all((fin / f"{t}.meta.json").exists() for t in TASKS):
        print(f"{arm}: incomplete, not scored")
        continue
    for t in TASKS:
        m = json.loads((fin / f"{t}.meta.json").read_text())
        assert m["read_gate_verdicts"] == [ARMS[arm]["verdict"]], (arm, t, m["read_gate_verdicts"])
    out_csv = ROOT / "results" / f"scores_{arm}.csv"
    run([PY, "-m", "modules.evaluation.longbench_scoring", "--predictions_dir", fin, "--out_csv", out_csv],
        f"score_{arm}")
    with open(out_csv) as fh:
        rows = {r["dataset"]: r for r in csv.DictReader(fh)}
    missing = [t for t in TASKS if t not in rows]
    assert not missing, f"{arm}: scorer did not score {missing}"
    for t in TASKS:
        assert int(rows[t]["num_examples"]) == EXPECT_N[t], (arm, t, rows[t])
    SCORES[arm] = {t: float(rows[t]["score"]) for t in TASKS}

if SCORES:
    allw = [(GEOM[t]["n"], GEOM[t]) for t in TASKS]
    tot = sum(n for n, _ in allw)
    mem = {"fullkv": "100%",
           "qevict_b010": (f"10% (nominal {100 * sum(n * s['nominal_mean'] for n, s in allw) / tot:.1f}%, "
                           f"physical {100 * sum(n * s['physical_mean'] for n, s in allw) / tot:.1f}%)")}
    head = ["Method", "KV memory"] + [LABEL[t] for t in TASKS] + ["Avg (11)"]
    table = []
    for arm in ("fullkv", "qevict_b010"):
        if arm in SCORES:
            v = [SCORES[arm][t] for t in TASKS]
            table.append([ARMS[arm]["label"], mem[arm]] + [f"{x:.2f}" for x in v] + [f"{statistics.mean(v):.2f}"])
    md_lines = ["| " + " | ".join(head) + " |", "|" + "---|" * len(head)] + ["| " + " | ".join(r) + " |" for r in table]
    (ROOT / "results" / "table_b010.md").write_text("\n".join(md_lines) + "\n")
    with open(ROOT / "results" / "table_b010.csv", "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(head)
        w.writerows(table)
    tex = [" & ".join([r[0], r[1].split(" ")[0].replace("%", r"\%")] + r[2:]) + r" \\" for r in table]
    (ROOT / "results" / "table_b010.tex").write_text("\n".join(tex) + "\n")
    print("\n".join(md_lines))
    print("\nLaTeX rows:\n" + "\n".join(tex))
    print(f"\nwritten to {ROOT / 'results'}")

## 10. Before quoting

* Every QEvict sidecar says `gated`, and the merged ones list every chunk's verdict (`read_gate_verdicts`).
  `scripts/check_read_gate.py <DRIVE_ROOT>/qevict_b010/final` re-checks them at any time.
* Quote the commit (`manifest.json`), the checkpoint revision and the stack with the numbers.
* State the MLA mapping (top of this notebook) in the paper's text: one shared latent "KV head" with rep = 16; the
  latent stored once in INT2 with per-channel grids (the paper's per-token value rule does not apply because the value
  is the key's latent); the latent byte accounting; the model's own softmax scale.
* State the 31,500-token middle truncation (THUDM's 32k-model convention), which the Llama rows did not need.
* `results/geometry_b010.json` gives, per task, how many examples had no INT2 tier at this budget and the nominal
  and physical KV-memory ratios.